# abismal GUI — minimal run panel

The full notebook needs ~185 s per epoch, and several epochs before a refinement
result appears. That is a long wait to look at a layout.

This drives the **same `AbismalRunner`** through the replay stub the test suite
uses — a real subprocess replaying a recorded run, with per-epoch results — so
the whole panel appears in a few seconds. Nothing is trained.

In [ ]:
# Two steps, and both before any abismal import.
#
# The first resolves dependencies -- on a fresh runtime nothing is installed yet,
# and --no-deps alone leaves abismal importable but broken on reciprocalspaceship.
# The second forces the code itself, because pip will otherwise skip a package it
# considers satisfied, and this notebook exists to look at current code.
%pip install -q "abismal[gui] @ git+https://github.com/rs-station/abismal@gui"
%pip install -q --force-reinstall --no-deps "abismal[gui] @ git+https://github.com/rs-station/abismal@gui"

import os, urllib.request

REF = "gui"
ROOT = "/content/abismal_replay"
RAW = f"https://raw.githubusercontent.com/rs-station/abismal/{REF}"

# The stub lives in the test tree, which is not packaged. gui_harness resolves
# these relative to itself, so the two directories are mirrored under one root.
for path, executable in {
    "devtools/gui_harness.py": False,
    "tests/gui/replay/abismal": True,      # the runner execs it
    "tests/gui/replay/console.log": False,
    "tests/gui/replay/history.csv": False,
}.items():
    destination = os.path.join(ROOT, path)
    os.makedirs(os.path.dirname(destination), exist_ok=True)
    urllib.request.urlretrieve(f"{RAW}/{path}", destination)
    if executable:
        os.chmod(destination, 0o755)

from abismal.gui._build import describe_build
print(describe_build(branch="colab-minimal"))

In [ ]:
import os, shutil, sys

# Start from an empty output directory. The runner shows the *latest* results it
# can find, which is right for a real job -- reopening a finished run should not
# replay its history -- but it means a rerun here would find the previous run's
# last epoch still on disk and jump straight to it, so the viewer and the peak
# plot would never appear to start over.
shutil.rmtree("/content/_replay_out", ignore_errors=True)

sys.path.insert(0, f"{ROOT}/devtools")
os.environ["PATH"] = f"{ROOT}/tests/gui/replay:" + os.environ["PATH"]

import gui_harness as H

# delay paces the epochs: slow enough to watch the progress bar move, fast
# enough that the run is over in seconds.
runner = H.start_replay(
    "/content/_replay_out",
    has_phenix=True,
    total_epochs=8,
    results=H.make_results_template("/content/_replay_template"),
    delay=0.8,
)

# Let the cell grow rather than scroll. Without this the panel is taller than
# Colab's output box and you land below the progress bar and training history,
# which reads as them failing to render.
from abismal.gui._launch import no_output_scroll

no_output_scroll()

# Displayed without waiting, so the panel updates from the runner's background
# threads -- which on Colab is the part that needs the poll loop working.
runner.to_widget()